In [1]:
"""
=============================================================================
  SOCINT Threat Intelligence – XGBoost Classifier  |  End-to-End Pipeline
=============================================================================
  Dataset  : socint_alerts.csv  (50 000 rows × 18 cols)
  Task     : 4-class ordinal threat classification
               0 = LOW | 1 = MODERATE | 2 = HIGH | 3 = CRITICAL
  Model    : XGBoost (multi:softprob) tuned with Optuna (Bayesian search)
  Output   : best_xgb_socint.json  (XGBoost native format, portable)
             label_encoder.pkl     (maps int → threat string)
             feature_columns.pkl   (ordered list used at inference time)
=============================================================================
"""

'\n=============================================================================\n  SOCINT Threat Intelligence – XGBoost Classifier  |  End-to-End Pipeline\n=============================================================================\n  Dataset  : socint_alerts.csv  (50 000 rows × 18 cols)\n  Task     : 4-class ordinal threat classification\n               0 = LOW | 1 = MODERATE | 2 = HIGH | 3 = CRITICAL\n  Model    : XGBoost (multi:softprob) tuned with Optuna (Bayesian search)\n  Output   : best_xgb_socint.json  (XGBoost native format, portable)\n             label_encoder.pkl     (maps int → threat string)\n             feature_columns.pkl   (ordered list used at inference time)\n=============================================================================\n'

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import pickle, json, os

from sklearn.model_selection   import StratifiedKFold, cross_val_score
from sklearn.preprocessing     import LabelEncoder
from sklearn.metrics           import (classification_report,
                                        confusion_matrix,
                                        ConfusionMatrixDisplay,
                                        f1_score, accuracy_score)
from sklearn.inspection        import permutation_importance

import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)   # silence Optuna spam

import matplotlib
matplotlib.use("Agg")   # headless – no display required
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# 1.  CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CSV_PATH   = "/Users/amitsingh/ML_Projects/WarfareAI/datasets/socint_alerts.csv"
OUT_DIR    = "./outputs"
SEED       = 42
N_SPLITS   = 5            # stratified k-fold splits
N_TRIALS   = 60           # Optuna Bayesian-search iterations (raise for more)
TEST_SIZE  = 0.20         # hold-out fraction

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)

In [5]:

# ─────────────────────────────────────────────────────────────────────────────
# 2.  DATA LOADING & SANITY CHECK
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("  SOCINT XGBoost  –  Training Pipeline")
print("=" * 65)

df = pd.read_csv(CSV_PATH)
print(f"\n[DATA]  Shape: {df.shape}")
print(f"[DATA]  Columns: {df.columns.tolist()}")
print(f"[DATA]  Nulls : {df.isnull().sum().sum()}  (expected 0)")

# --- target sanity ---
# threat_label_int already encodes ordinal severity correctly:
#   LOW=0  MODERATE=1  HIGH=2  CRITICAL=3
print("\n[TARGET] class distribution:")
print(df.groupby(["threat_label_int","threat_label"]).size()
        .reset_index(name="count").to_string(index=False))

  SOCINT XGBoost  –  Training Pipeline

[DATA]  Shape: (50000, 18)
[DATA]  Columns: ['timestamp', 'region', 'n_protests', 'n_propaganda', 'n_informants', 'n_total_alerts', 'n_high_sev', 'n_med_sev', 'n_low_sev', 'high_sev_ratio', 'propaganda_ratio', 'informant_ratio', 'mean_severity_score', 'alerts_near_border', 'n_active_scenarios', 'socint_score', 'threat_label', 'threat_label_int']
[DATA]  Nulls : 0  (expected 0)

[TARGET] class distribution:
 threat_label_int threat_label  count
                0          LOW  14038
                1     MODERATE  14914
                2         HIGH  13515
                3     CRITICAL   7533


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.  FEATURE ENGINEERING
#
#     The raw CSV already ships 12 model-ready features.  We add four groups
#     of engineered signals on top:
#
#       A) Temporal  – hour-of-day, day-of-week, is_weekend extracted from
#                      the ISO-8601 `timestamp` field.  Threat cadence often
#                      peaks at night or on weekends.
#
#       B) Region    – ordinal-encoded from the `region` string.
#                      We do NOT one-hot-encode; XGBoost handles ordinal
#                      integers natively with no dimensionality blowup.
#
#       C) Ratio     – weighted_sev_ratio: re-applies the simulator's own
#                      severity weights (HIGH=1.0 MED=0.5 LOW=0.2) and
#                      normalises by total alerts.  Provides a single scalar
#                      that aligns directly with the threat scoring formula.
#
#       D) Interaction – alert_pressure: (n_total_alerts × mean_severity_score)
#                        combined signal of volume AND severity.
#                        high_sev_x_border: high-severity alerts that are also
#                        near-border amplifiers.
#                        propaganda_x_informant: co-occurrence ratio; signals
#                        coordinated influence + ground-level intelligence.
# ─────────────────────────────────────────────────────────────────────────────

print("\n[FEAT]  Engineering features …")

# A) Temporal features
df["timestamp"]  = pd.to_datetime(df["timestamp"])
df["hour"]       = df["timestamp"].dt.hour                  # 0–23
df["day_of_week"]= df["timestamp"].dt.dayofweek             # 0=Mon … 6=Sun
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)     # binary flag
df["month"]      = df["timestamp"].dt.month                 # 1–12 seasonal

# B) Region – ordinal integer encoding
region_le = LabelEncoder()
df["region_enc"] = region_le.fit_transform(df["region"])
encoded_regions = dict(zip(region_le.classes_, region_le.transform(region_le.classes_)))
print(f"[FEAT]  Regions encoded: {encoded_regions}")

# C) Severity-weight ratio aligned with simulator weights
#    HIGH=1.0, MED=0.5, LOW=0.2  (same weights used in threat scoring)
SEV_W_HIGH = 1.0
SEV_W_MED  = 0.5
SEV_W_LOW  = 0.2

df["weighted_sev_score"] = (
    df["n_high_sev"] * SEV_W_HIGH +
    df["n_med_sev"]  * SEV_W_MED  +
    df["n_low_sev"]  * SEV_W_LOW
)
# Normalise by total alerts (avoid divide-by-zero with eps)
df["weighted_sev_ratio"] = (
    df["weighted_sev_score"] /
    (df["n_total_alerts"] + 1e-9)
)

# D) Interaction features
df["alert_pressure"]         = df["n_total_alerts"] * df["mean_severity_score"]
df["high_sev_x_border"]      = df["n_high_sev"]     * df["alerts_near_border"]
df["propaganda_x_informant"] = df["propaganda_ratio"] * df["informant_ratio"]
# Scenario multiplier: active scenarios amplify every other signal
df["scenario_amplifier"]     = (df["n_active_scenarios"] + 1) * df["mean_severity_score"]

print("[FEAT]  New columns added:")
new_feats = ["hour","day_of_week","is_weekend","month","region_enc",
             "weighted_sev_score","weighted_sev_ratio",
             "alert_pressure","high_sev_x_border",
             "propaganda_x_informant","scenario_amplifier"]
for f in new_feats:
    print(f"         + {f}")



[FEAT]  Engineering features …
[FEAT]  Regions encoded: {'india-arunachal': 0, 'india-bangladesh': 1, 'india-china': 2, 'india-pak': 3}
[FEAT]  New columns added:
         + hour
         + day_of_week
         + is_weekend
         + month
         + region_enc
         + weighted_sev_score
         + weighted_sev_ratio
         + alert_pressure
         + high_sev_x_border
         + propaganda_x_informant
         + scenario_amplifier


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.  FEATURE MATRIX  &  TARGET VECTOR
#
#     We deliberately DROP:
#       • timestamp   – already decomposed above; raw datetime unusable by XGB
#       • region      – replaced by region_enc
#       • threat_label – string version of the target (already have int)
#       • socint_score – this is derived from the SAME formula that produced
#                        the labels; keeping it would be label leakage.
# ─────────────────────────────────────────────────────────────────────────────

DROP_COLS = ["timestamp", "region", "threat_label",
             "socint_score",         # <── leakage guard
             "weighted_sev_score"]   # intermediate; keep the ratio instead

FEATURE_COLS = [c for c in df.columns
                if c not in DROP_COLS + ["threat_label_int"]]

print(f"\n[FEAT]  Final feature set ({len(FEATURE_COLS)} features):")
for i, f in enumerate(FEATURE_COLS, 1):
    print(f"        {i:02d}. {f}")

X = df[FEATURE_COLS].values
y = df["threat_label_int"].values      # 0 / 1 / 2 / 3  (already ordinal)

print(f"\n[DATA]  X shape: {X.shape}   y shape: {y.shape}")
print(f"[DATA]  Classes: {np.unique(y)}")



[FEAT]  Final feature set (23 features):
        01. n_protests
        02. n_propaganda
        03. n_informants
        04. n_total_alerts
        05. n_high_sev
        06. n_med_sev
        07. n_low_sev
        08. high_sev_ratio
        09. propaganda_ratio
        10. informant_ratio
        11. mean_severity_score
        12. alerts_near_border
        13. n_active_scenarios
        14. hour
        15. day_of_week
        16. is_weekend
        17. month
        18. region_enc
        19. weighted_sev_ratio
        20. alert_pressure
        21. high_sev_x_border
        22. propaganda_x_informant
        23. scenario_amplifier

[DATA]  X shape: (50000, 23)   y shape: (50000,)
[DATA]  Classes: [0 1 2 3]


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 5.  TRAIN / TEST SPLIT  (stratified so every class is represented)
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size   = TEST_SIZE,
    random_state= SEED,
    stratify    = y          # preserves class ratios in both splits
)

print(f"\n[SPLIT] Train: {X_train.shape[0]}  Test: {X_test.shape[0]}")

# Class-weight calculation for imbalance handling
# XGBoost multi-class uses `sample_weight` not `scale_pos_weight`
from sklearn.utils.class_weight import compute_sample_weight
sample_weights_train = compute_sample_weight("balanced", y_train)




[SPLIT] Train: 40000  Test: 10000


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.  BASELINE MODEL  (default hyperparams – sets performance floor)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[BASE]  Training baseline XGBoost …")

baseline = xgb.XGBClassifier(
    objective        = "multi:softprob",
    num_class        = 4,
    eval_metric      = "mlogloss",
    use_label_encoder= False,
    random_state     = SEED,
    n_jobs           = -1
)
baseline.fit(X_train, y_train, sample_weight=sample_weights_train)
base_acc = accuracy_score(y_test, baseline.predict(X_test))
base_f1  = f1_score(y_test, baseline.predict(X_test), average="macro")
print(f"[BASE]  Accuracy={base_acc:.4f}   Macro-F1={base_f1:.4f}")


[BASE]  Training baseline XGBoost …
[BASE]  Accuracy=0.9332   Macro-F1=0.9339


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# 7.  HYPERPARAMETER TUNING  –  Optuna (Tree-structured Parzen Estimator)
#
#     We search over the six most impactful XGBoost knobs:
#       n_estimators     – number of boosting rounds
#       max_depth        – tree depth  (bias/variance tradeoff)
#       learning_rate    – shrinkage (η); lower = needs more rounds
#       subsample        – row sampling per tree  (stochastic boosting)
#       colsample_bytree – feature sampling per tree  (like RF bagging)
#       min_child_weight – minimum sum of instance weight in a leaf
#                          (controls over-fitting on minority classes)
#       gamma            – min loss reduction to make a further partition
#       reg_alpha        – L1 regularisation (feature sparsity)
#       reg_lambda       – L2 regularisation (weight smoothing)
#
#     Objective: maximise stratified-CV macro-F1 on training data.
#     Macro-F1 is preferred over accuracy because the dataset has moderate
#     class imbalance (CRITICAL << LOW).
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n[TUNE]  Running Optuna ({N_TRIALS} trials, {N_SPLITS}-fold CV) …")

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

def objective(trial: optuna.Trial) -> float:
    """Return mean CV macro-F1 for a candidate parameter set."""
    params = {
        "objective"        : "multi:softprob",
        "num_class"        : 4,
        "eval_metric"      : "mlogloss",
        "use_label_encoder": False,
        "random_state"     : SEED,
        "n_jobs"           : -1,
        # ── searched hyperparameters ──────────────────────────────────────
        "n_estimators"    : trial.suggest_int  ("n_estimators",     100, 800),
        "max_depth"       : trial.suggest_int  ("max_depth",          3,  10),
        "learning_rate"   : trial.suggest_float("learning_rate",    0.01, 0.30, log=True),
        "subsample"       : trial.suggest_float("subsample",         0.5,  1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree",  0.4,  1.0),
        "min_child_weight": trial.suggest_int  ("min_child_weight",    1,  10),
        "gamma"           : trial.suggest_float("gamma",             0.0,  5.0),
        "reg_alpha"       : trial.suggest_float("reg_alpha",         0.0,  2.0),
        "reg_lambda"      : trial.suggest_float("reg_lambda",        0.5,  5.0),
    }
    model = xgb.XGBClassifier(**params)
    scores = cross_val_score(
        model, X_train, y_train,
        cv             = skf,
        scoring        = "f1_macro",
        fit_params     = {"sample_weight": sample_weights_train},
        n_jobs         = 1         # inner loop already uses -1 for XGB
    )
    return scores.mean()


study = optuna.create_study(
    direction   = "maximize",
    sampler     = optuna.samplers.TPESampler(seed=SEED),   # Bayesian TPE
    pruner      = optuna.pruners.MedianPruner()            # stop bad trials early
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

best_params = study.best_params
best_cv_f1  = study.best_value
print(f"\n[TUNE]  Best CV macro-F1 : {best_cv_f1:.4f}")
print("[TUNE]  Best hyperparams :")
for k, v in best_params.items():
    print(f"         {k:<22s} = {v}")



[TUNE]  Running Optuna (60 trials, 5-fold CV) …

[TUNE]  Best CV macro-F1 : 0.9339
[TUNE]  Best hyperparams :
         n_estimators           = 683
         max_depth              = 6
         learning_rate          = 0.04085168416171183
         subsample              = 0.8217898748355169
         colsample_bytree       = 0.8269274308145127
         min_child_weight       = 5
         gamma                  = 1.7211483298480752
         reg_alpha              = 0.8149806033625668
         reg_lambda             = 1.791326906288527


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 8.  FINAL MODEL TRAINING  (full training set + best params)
#
#     We use early_stopping_rounds on the held-out test set ONLY to
#     determine the optimal n_estimators; the test labels are NOT used
#     to select hyperparameters (those came from CV above).
# ─────────────────────────────────────────────────────────────────────────────
print("\n[TRAIN] Fitting final model with best hyperparams …")

# Override n_estimators with a higher ceiling; early stopping finds best round
best_params_final = {**best_params,
                     "n_estimators"     : 2000,      # ceiling; early stop cuts it
                     "objective"        : "multi:softprob",
                     "num_class"        : 4,
                     "eval_metric"      : "mlogloss",
                     "use_label_encoder": False,
                     "random_state"     : SEED,
                     "n_jobs"           : -1}

final_model = xgb.XGBClassifier(**best_params_final)
final_model.fit(
    X_train, y_train,
    sample_weight       = sample_weights_train,
    eval_set            = [(X_test, y_test)],
    early_stopping_rounds = 30,          # stop if no improvement for 30 rounds
    verbose             = False
)
print(f"[TRAIN] Best iteration: {final_model.best_iteration}")



[TRAIN] Fitting final model with best hyperparams …
[TRAIN] Best iteration: 379


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# 9.  EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
y_pred      = final_model.predict(X_test)
y_pred_prob = final_model.predict_proba(X_test)      # shape (N, 4)

acc     = accuracy_score(y_test, y_pred)
f1_mac  = f1_score(y_test, y_pred, average="macro")
f1_wt   = f1_score(y_test, y_pred, average="weighted")

LABEL_NAMES = ["LOW", "MODERATE", "HIGH", "CRITICAL"]

print("\n" + "=" * 55)
print("  EVALUATION ON HELD-OUT TEST SET")
print("=" * 55)
print(f"  Accuracy        : {acc:.4f}")
print(f"  Macro-F1        : {f1_mac:.4f}")
print(f"  Weighted-F1     : {f1_wt:.4f}")
print(f"  Baseline Acc    : {base_acc:.4f}  (Δ = {acc - base_acc:+.4f})")
print(f"  Baseline Mac-F1 : {base_f1:.4f}  (Δ = {f1_mac - base_f1:+.4f})")
print()
print(classification_report(y_test, y_pred, target_names=LABEL_NAMES))




  EVALUATION ON HELD-OUT TEST SET
  Accuracy        : 0.9318
  Macro-F1        : 0.9325
  Weighted-F1     : 0.9316
  Baseline Acc    : 0.9332  (Δ = -0.0014)
  Baseline Mac-F1 : 0.9339  (Δ = -0.0014)

              precision    recall  f1-score   support

         LOW       0.95      0.97      0.96      2808
    MODERATE       0.92      0.91      0.92      2983
        HIGH       0.93      0.89      0.91      2703
    CRITICAL       0.92      0.96      0.94      1506

    accuracy                           0.93     10000
   macro avg       0.93      0.93      0.93     10000
weighted avg       0.93      0.93      0.93     10000



In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# 10.  PLOTS
# ─────────────────────────────────────────────────────────────────────────────
print("[PLOT] Generating diagnostic plots …")

# ── palette ──────────────────────────────────────────────────────────────────
BG   = "#0d1117"
ACC  = "#00e5ff"
WARN = "#ff6b35"
MID  = "#4ecdc4"
MILD = "#1a2332"

plt.rcParams.update({
    "figure.facecolor" : BG, "axes.facecolor"  : MILD,
    "axes.edgecolor"   : ACC, "axes.labelcolor" : ACC,
    "xtick.color"      : ACC, "ytick.color"     : ACC,
    "text.color"       : ACC, "grid.color"      : "#1f2d3d",
    "font.family"      : "monospace"
})

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("SOCINT XGBoost  –  Threat Classifier Report",
             color=ACC, fontsize=14, fontweight="bold")
fig.patch.set_facecolor(BG)

# ── 10-A  Confusion matrix ───────────────────────────────────────────────────
ax = axes[0, 0]
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrRd",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            linewidths=0.5, linecolor="#2a3a4a", ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Confusion Matrix", color=ACC, pad=10)
ax.set_xlabel("Predicted", color=ACC)
ax.set_ylabel("True", color=ACC)

# ── 10-B  Feature importances  (gain) ────────────────────────────────────────
ax = axes[0, 1]
importances = final_model.feature_importances_
feat_imp = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=True)
colors    = [WARN if v > feat_imp.median() else MID for v in feat_imp]
feat_imp.plot(kind="barh", ax=ax, color=colors, edgecolor="none")
ax.set_title("Feature Importances (Gain)", color=ACC, pad=10)
ax.set_xlabel("Importance", color=ACC)
ax.axvline(feat_imp.median(), color="white", linestyle="--", alpha=0.5, linewidth=0.8)

# ── 10-C  Class probability distribution per true label ──────────────────────
ax = axes[1, 0]
for cls_idx, cls_name in enumerate(LABEL_NAMES):
    mask    = y_test == cls_idx
    probs   = y_pred_prob[mask, cls_idx]
    ax.hist(probs, bins=30, alpha=0.65, label=cls_name,
            histtype="stepfilled", linewidth=1.2)
ax.set_title("Predicted Probability (correct class)", color=ACC, pad=10)
ax.set_xlabel("P(true class)", color=ACC)
ax.set_ylabel("Frequency", color=ACC)
ax.legend(facecolor=MILD, edgecolor=ACC, labelcolor=ACC)

# ── 10-D  Optuna optimisation history ────────────────────────────────────────
ax = axes[1, 1]
trial_vals = [t.value for t in study.trials if t.value is not None]
best_so_far = np.maximum.accumulate(trial_vals)
ax.plot(trial_vals, color=MID, alpha=0.4, linewidth=0.8, label="Trial F1")
ax.plot(best_so_far, color=WARN, linewidth=2, label="Best so far")
ax.set_title("Optuna Optimisation History", color=ACC, pad=10)
ax.set_xlabel("Trial", color=ACC)
ax.set_ylabel("Macro-F1 (CV)", color=ACC)
ax.legend(facecolor=MILD, edgecolor=ACC, labelcolor=ACC)

plt.tight_layout()
plot_path = os.path.join(OUT_DIR, "xgb_evaluation.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"[PLOT] Saved  → {plot_path}")


[PLOT] Generating diagnostic plots …
[PLOT] Saved  → ./outputs/xgb_evaluation.png


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# 11.  PERMUTATION IMPORTANCE  (model-agnostic, more reliable than gain)
#
#      Permutation importance shuffles one feature column at a time on the
#      test set and measures the drop in macro-F1. This catches features
#      that are not split-frequent (low gain) but are still informative.
# ─────────────────────────────────────────────────────────────────────────────
print("[PERM]  Computing permutation importance …")

perm = permutation_importance(
    final_model, X_test, y_test,
    n_repeats  = 10,
    random_state = SEED,
    scoring    = "f1_macro"
)
perm_df = pd.DataFrame({
    "feature"  : FEATURE_COLS,
    "mean_drop": perm.importances_mean,
    "std_drop" : perm.importances_std
}).sort_values("mean_drop", ascending=False)

print("\n[PERM]  Top-10 features by permutation importance:")
print(perm_df.head(10).to_string(index=False))

# Save permutation importance chart
fig2, ax2 = plt.subplots(figsize=(10, 7))
fig2.patch.set_facecolor(BG)
ax2.set_facecolor(MILD)
top15 = perm_df.head(15).sort_values("mean_drop")
ax2.barh(top15["feature"], top15["mean_drop"],
         xerr=top15["std_drop"], color=WARN,
         ecolor=ACC, alpha=0.85, capsize=3)
ax2.set_title("Permutation Importance (top 15, macro-F1 drop)", color=ACC)
ax2.set_xlabel("Mean decrease in macro-F1", color=ACC)
ax2.tick_params(colors=ACC)
for spine in ax2.spines.values():
    spine.set_edgecolor(ACC)
plt.tight_layout()
perm_path = os.path.join(OUT_DIR, "xgb_permutation_importance.png")
plt.savefig(perm_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"[PERM]  Saved  → {perm_path}")


[PERM]  Computing permutation importance …

[PERM]  Top-10 features by permutation importance:
               feature  mean_drop  std_drop
    scenario_amplifier   0.182640  0.003469
            n_high_sev   0.164441  0.003843
    n_active_scenarios   0.155919  0.001582
    alerts_near_border   0.091381  0.001864
     high_sev_x_border   0.040064  0.001556
        alert_pressure   0.026556  0.001215
   mean_severity_score   0.006995  0.001063
             n_med_sev   0.003501  0.000833
        n_total_alerts   0.001841  0.000497
propaganda_x_informant   0.001128  0.000350
[PERM]  Saved  → ./outputs/xgb_permutation_importance.png


In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# 12.  CROSS-VALIDATION  (final model, 5-fold, macro-F1)
#
#      Re-run CV with the tuned estimator to get a bias-corrected performance
#      estimate that is independent of the train/test split.
# ─────────────────────────────────────────────────────────────────────────────
print("\n[CV]    Running 5-fold CV on full dataset with best params …")

cv_model = xgb.XGBClassifier(
    **{k: v for k, v in best_params_final.items()
       if k not in ["n_estimators"]},
    n_estimators = final_model.best_iteration + 1   # use optimal stopping point
)

cv_scores = cross_val_score(
    cv_model, X, y,
    cv      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED),
    scoring = "f1_macro",
    n_jobs  = -1
)
print(f"[CV]    Fold scores : {np.round(cv_scores, 4)}")
print(f"[CV]    Mean ± Std  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")



[CV]    Running 5-fold CV on full dataset with best params …
[CV]    Fold scores : [0.9397 0.9332 0.9356 0.9366 0.9294]
[CV]    Mean ± Std  : 0.9349 ± 0.0035


In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# 13.  MODEL ARTEFACT SAVE
# ─────────────────────────────────────────────────────────────────────────────
model_path   = os.path.join(OUT_DIR, "best_xgb_socint.json")
le_path      = os.path.join(OUT_DIR, "label_encoder.pkl")
feat_path    = os.path.join(OUT_DIR, "feature_columns.pkl")
metrics_path = os.path.join(OUT_DIR, "training_metrics.json")

# XGBoost native JSON format (portable, version-safe)
final_model.save_model(model_path)

# Label map  {0:"LOW", 1:"MODERATE", 2:"HIGH", 3:"CRITICAL"}
label_map = {i: lbl for i, lbl in enumerate(LABEL_NAMES)}
with open(le_path, "wb") as f:
    pickle.dump(label_map, f)

# Ordered feature list required for inference
with open(feat_path, "wb") as f:
    pickle.dump(FEATURE_COLS, f)

# Human-readable metrics summary
metrics = {
    "accuracy"         : round(float(acc),    4),
    "macro_f1"         : round(float(f1_mac), 4),
    "weighted_f1"      : round(float(f1_wt),  4),
    "cv_macro_f1_mean" : round(float(cv_scores.mean()), 4),
    "cv_macro_f1_std"  : round(float(cv_scores.std()),  4),
    "best_iteration"   : int(final_model.best_iteration),
    "n_features"       : len(FEATURE_COLS),
    "best_params"      : best_params
}
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\n[SAVE]  Model      → {model_path}")
print(f"[SAVE]  Label map  → {le_path}")
print(f"[SAVE]  Features   → {feat_path}")
print(f"[SAVE]  Metrics    → {metrics_path}")



[SAVE]  Model      → ./outputs/best_xgb_socint.json
[SAVE]  Label map  → ./outputs/label_encoder.pkl
[SAVE]  Features   → ./outputs/feature_columns.pkl
[SAVE]  Metrics    → ./outputs/training_metrics.json


In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# 14.  INFERENCE HELPER  (copy this block into your scoring pipeline)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  INFERENCE DEMO  (single sample)")
print("=" * 55)

def predict_threat(raw_row: pd.Series,
                   model: xgb.XGBClassifier,
                   feature_cols: list,
                   label_map: dict) -> dict:
    """
    Given a raw alert row (with the same schema as socint_alerts.csv),
    return the predicted threat label and per-class probabilities.

    Parameters
    ----------
    raw_row     : pd.Series  – one row from the raw CSV
    model       : trained XGBClassifier
    feature_cols: list of column names in order (from feature_columns.pkl)
    label_map   : dict {int → str}  (from label_encoder.pkl)

    Returns
    -------
    dict with keys: predicted_class, predicted_label, probabilities
    """
    # Replicate the same feature engineering as training
    row = raw_row.copy()
    ts  = pd.to_datetime(row["timestamp"])
    row["hour"]        = ts.hour
    row["day_of_week"] = ts.dayofweek
    row["is_weekend"]  = int(ts.dayofweek >= 5)
    row["month"]       = ts.month

    # Region – use the same encoder; unseen regions map to -1 (OOV guard)
    try:
        row["region_enc"] = region_le.transform([row["region"]])[0]
    except ValueError:
        row["region_enc"] = -1

    row["weighted_sev_score"] = (
        row["n_high_sev"] * SEV_W_HIGH +
        row["n_med_sev"]  * SEV_W_MED  +
        row["n_low_sev"]  * SEV_W_LOW
    )
    row["weighted_sev_ratio"] = (
        row["weighted_sev_score"] / (row["n_total_alerts"] + 1e-9)
    )
    row["alert_pressure"]         = row["n_total_alerts"] * row["mean_severity_score"]
    row["high_sev_x_border"]      = row["n_high_sev"]     * row["alerts_near_border"]
    row["propaganda_x_informant"] = row["propaganda_ratio"] * row["informant_ratio"]
    row["scenario_amplifier"]     = (row["n_active_scenarios"] + 1) * row["mean_severity_score"]

    X_infer  = pd.DataFrame([row])[feature_cols].values
    pred_cls = int(model.predict(X_infer)[0])
    pred_prb = model.predict_proba(X_infer)[0]

    return {
        "predicted_class" : pred_cls,
        "predicted_label" : label_map[pred_cls],
        "probabilities"   : {label_map[i]: round(float(p), 4)
                             for i, p in enumerate(pred_prb)}
    }


# Demo on a random test sample
demo_row = df.iloc[42]
result   = predict_threat(demo_row, final_model, FEATURE_COLS, label_map)
print(f"  True label  : {demo_row['threat_label']}")
print(f"  Prediction  : {result['predicted_label']}")
print(f"  Probabilities:")
for lbl, p in result["probabilities"].items():
    bar = "█" * int(p * 40)
    print(f"    {lbl:<10s} {p:.4f}  {bar}")



  INFERENCE DEMO  (single sample)
  True label  : MODERATE
  Prediction  : MODERATE
  Probabilities:
    LOW        0.0005  
    MODERATE   0.8666  ██████████████████████████████████
    HIGH       0.1326  █████
    CRITICAL   0.0003  


In [20]:

# ─────────────────────────────────────────────────────────────────────────────
# 15.  SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  TRAINING COMPLETE")
print("=" * 55)
print(f"  Accuracy         : {acc:.4f}")
print(f"  Macro-F1 (test)  : {f1_mac:.4f}")
print(f"  Macro-F1 (5-CV)  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  Best iteration   : {final_model.best_iteration}")
print(f"  Features used    : {len(FEATURE_COLS)}")
print(f"  Output artefacts : {OUT_DIR}/")
print("=" * 55)


  TRAINING COMPLETE
  Accuracy         : 0.9318
  Macro-F1 (test)  : 0.9325
  Macro-F1 (5-CV)  : 0.9349 ± 0.0035
  Best iteration   : 379
  Features used    : 23
  Output artefacts : ./outputs/
